# ε-Insensitive Support Vector Regression

Support vector regression fits a function whose deviations of at most $\epsilon$ incur no loss. With slack variables $\xi_i,\xi_i^*\ge0$, the primal objective is

$$\min_{w,b,\xi,\xi^*}\;\frac12\|w\|_2^2+C\sum_i(\xi_i+\xi_i^*).$$

The constraints are $y_i-f(x_i)\le\epsilon+\xi_i$ and $f(x_i)-y_i\le\epsilon+\xi_i^*$. The dual coefficients satisfy $0\le\alpha_i,\alpha_i^*\le C_i$ and $\sum_i(\alpha_i-\alpha_i^*)=0$, where $C_i$ incorporates optional observation weights.


## Implementation

The solver uses feasible two-variable updates along signed-gradient directions and caches kernel responses. The KKT gap controls termination, while the primal and dual objectives quantify numerical convergence. The implementation uses Python's standard library.

Optional progress and trace arguments record kernel construction, optimization checks and objective values during training.


In [ ]:
from array import array
from dataclasses import dataclass
from math import exp, inf, isfinite
from statistics import median


@dataclass(frozen=True)
class Geometry:
    form: str = 'radial'
    radial_width: float = 1.0
    polynomial_order: int = 2

    def __post_init__(self):
        if self.form not in ('linear', 'radial', 'polynomial'):
            raise ValueError('Unknown kernel geometry.')
        if not isfinite(self.radial_width) or self.radial_width <= 0:
            raise ValueError('The radial width must be positive and finite.')
        if not isinstance(self.polynomial_order, int) or self.polynomial_order < 1:
            raise ValueError('Polynomial order must be a positive integer.')


def _checked_rows(observations, outcomes):
    records = tuple(tuple(float(v) for v in row) for row in observations)
    if not records or len(records) != len(outcomes) or not records[0]:
        raise ValueError('Nonempty matched observations/outcomes required.')
    width = len(records[0])
    if any(len(row) != width for row in records):
        raise ValueError('Nonuniform feature dimensions.')
    if any(not isfinite(v) for row in records for v in row):
        raise ValueError('Nonfinite input features.')
    return records


def _compare_rows(a, b, geometry):
    if geometry.form == 'radial':
        squared_distance = sum((u - v) * (u - v) for u, v in zip(a, b))
        return exp(-squared_distance / (2 * geometry.radial_width ** 2))
    linear_product = sum(u * v for u, v in zip(a, b))
    if geometry.form == 'linear':
        return linear_product
    return (1 + linear_product) ** geometry.polynomial_order


def _packed_kernel(records, geometry, progress=None):
    """Symmetric row-major Gram table in 64-bit floating-point storage."""
    n = len(records)
    table = array('d', [0.0]) * (n * n)
    rows = range(n) if progress is None else progress(
        range(n), desc="SVR Gram matrix", unit="row")
    for left in rows:
        base = left * n
        for right in range(left, n):
            value = _compare_rows(records[left], records[right], geometry)
            if not isfinite(value):
                raise FloatingPointError('Nonfinite kernel entry.')
            table[base + right] = value
            table[right * n + left] = value
    return table


@dataclass(frozen=True)
class EpsilonTube:
    retained_examples: tuple
    signed_weights: tuple
    offset: float
    geometry: Geometry
    penalty: float
    tube_width: float
    updates: int
    stationarity_gap: float
    kkt_error: float
    primal_value: float
    dual_value: float
    relative_gap: float
    certified: bool


def _tube_multiplier_error(excess, amount, maximum):
    small = min(1e-7, maximum * 1e-6)
    if amount <= small:
        return max(0.0, excess)
    if amount >= maximum - small:
        return max(0.0, -excess)
    return abs(excess)


def construct_epsilon_tube(observations, targets, *,
                           geometry=Geometry('radial'),
                           penalty=1.0, tube_width=0.1,
                           precision=1e-3, update_budgets=50,
                           observation_importance=None, progress=None,
                           trace=None, trace_every=250):
    """Fit the epsilon-insensitive regression dual.

    Use signed feasible directions and cached kernel responses.
    """
    observations_y = tuple(float(v) for v in targets)
    records = _checked_rows(observations, observations_y)
    if not all(isfinite(v) for v in observations_y):
        raise ValueError('Nonfinite regression targets.')
    if (not isfinite(penalty) or penalty <= 0 or not isfinite(tube_width)
            or tube_width < 0 or precision <= 0 or update_budgets < 1):
        raise ValueError('Invalid regression optimization settings.')
    n = len(records)
    if observation_importance is None:
        caps = [penalty] * n
    else:
        importance = tuple(float(v) for v in observation_importance)
        if len(importance) != n or any(not isfinite(v) or v <= 0 for v in importance):
            raise ValueError('Observation importance must contain positive finite values.')
        caps = [penalty * v for v in importance]
    if not all(isfinite(v) for v in caps):
        raise ValueError('Unbounded importance-weighted penalties.')

    gram = _packed_kernel(records, geometry, progress=progress)
    doubled = 2 * n
    dual_multipliers = [0.0] * doubled
    limit = caps + caps
    direction = (1,) * n + (-1,) * n
    kernel_response = [0.0] * n
    diagonal = [gram[i * n + i] for i in range(n)]
    updates = 0
    max_updates = update_budgets * doubled
    required_gap = max(precision, 1e-7)
    near_zero = 1e-12
    gap = inf
    if trace is not None and trace_every < 1:
        raise ValueError('trace_every must be positive.')

    def record_trace(stationarity, intercept=None):
        # Slack variables make the primal point feasible for any finite intercept.
        coefficients = [dual_multipliers[i] - dual_multipliers[n + i] for i in range(n)]
        quadratic = sum(coefficients[i] * kernel_response[i] for i in range(n))
        objective = (
            -0.5 * quadratic
            + sum(coefficients[i] * observations_y[i] for i in range(n))
            - tube_width * sum(dual_multipliers)
        )
        if intercept is None:
            intercept = median(observations_y[i] - kernel_response[i] for i in range(n))
        upper = 0.5 * quadratic + sum(
            caps[i] * max(0., abs(observations_y[i] - kernel_response[i] - intercept) - tube_width)
            for i in range(n)
        )
        trace.append({
            'updates': updates,
            'stationarity_gap': stationarity,
            'dual': objective,
            'primal': upper,
            'gap': upper - objective,
        })

    attempts = range(max_updates + 1)
    if progress is not None:
        # Count KKT checks during optimization.
        attempts = progress((i for i in attempts), desc="SVR KKT checks", unit="check")
    for attempt in attempts:
        stationary = [kernel_response[j] + tube_width - observations_y[j] for j in range(n)]
        stationary += [kernel_response[j] - tube_width - observations_y[j] for j in range(n)]
        source = -1
        least = inf
        greatest = -inf
        for j in range(doubled):
            sign = direction[j]
            up_possible = (dual_multipliers[j] < limit[j] - near_zero) if sign == 1 else (dual_multipliers[j] > near_zero)
            down_possible = (dual_multipliers[j] > near_zero) if sign == 1 else (dual_multipliers[j] < limit[j] - near_zero)
            if up_possible and stationary[j] < least:
                least, source = stationary[j], j
            if down_possible and stationary[j] > greatest:
                greatest = stationary[j]
        gap = max(0.0, greatest - least)
        if trace is not None and updates % trace_every == 0 and gap > required_gap:
            record_trace(gap)
        if gap <= required_gap:
            break
        if source < 0 or attempt >= max_updates:
            raise RuntimeError(f'Epsilon tube failed stationarity: {gap:.6g} after {updates} updates.')
        from_index = source % n
        sign_source = direction[source]
        source_capacity = ((limit[source] - dual_multipliers[source]) if sign_source == 1
                           else dual_multipliers[source])
        chosen = -1
        best_step, best_reward = 0.0, -inf
        for sink in range(doubled):
            sign_sink = direction[sink]
            sink_capacity = (dual_multipliers[sink] if sign_sink == 1
                             else limit[sink] - dual_multipliers[sink])
            boundary = min(source_capacity, sink_capacity)
            if boundary <= near_zero:
                continue
            slope = stationary[sink] - least
            if slope <= 0:
                continue
            other = sink % n
            curvature = max(0.0, diagonal[from_index] + diagonal[other] - 2 * gram[from_index * n + other])
            change = min(boundary, slope / curvature) if curvature > 1e-14 else boundary
            reward = slope * change - 0.5 * curvature * change * change
            if reward > best_reward:
                best_reward, best_step, chosen = reward, change, sink
        if chosen < 0:
            raise RuntimeError('No feasible regression pair while stationarity is unsatisfied.')
        to_index = chosen % n
        dual_multipliers[source] += sign_source * best_step
        dual_multipliers[chosen] -= direction[chosen] * best_step
        origin_row = from_index * n
        destination_row = to_index * n
        for j in range(n):
            kernel_response[j] += best_step * (gram[origin_row + j] - gram[destination_row + j])
        updates += 1

    alpha = dual_multipliers[:n]
    alpha_conjugate = dual_multipliers[n:]
    signed = [alpha[j] - alpha_conjugate[j] for j in range(n)]

    possible_bias = []
    lower = -inf
    upper = inf
    for j in range(n):
        a, conjugate, cap = alpha[j], alpha_conjugate[j], caps[j]
        small = min(max(1e-7, precision ** 0.5), 0.1 * cap)
        low = observations_y[j] - tube_width - kernel_response[j]
        high = observations_y[j] + tube_width - kernel_response[j]
        if small < a < cap - small:
            possible_bias.append(low)
        if small < conjugate < cap - small:
            possible_bias.append(high)
        if a <= small:
            lower = max(lower, low)
        if a >= cap - small:
            upper = min(upper, low)
        if conjugate <= small:
            upper = min(upper, high)
        if conjugate >= cap - small:
            lower = max(lower, high)
    if possible_bias:
        offset = sum(possible_bias) / len(possible_bias)
    elif isfinite(lower) and isfinite(upper) and lower <= upper + 1e-8:
        offset = 0.5 * (lower + upper)
    else:
        raise RuntimeError('Inconsistent regression bias constraints.')

    inner = sum(signed[i] * kernel_response[i] for i in range(n))
    dual = -0.5 * inner + sum(signed[i] * observations_y[i] for i in range(n)) - tube_width * sum(dual_multipliers)
    residuals = [observations_y[i] - kernel_response[i] - offset for i in range(n)]
    primal = 0.5 * inner + sum(caps[i] * max(0.0, abs(residuals[i]) - tube_width) for i in range(n))
    error = max(max(_tube_multiplier_error(residuals[i] - tube_width, alpha[i], caps[i]),
                    _tube_multiplier_error(-residuals[i] - tube_width, alpha_conjugate[i], caps[i]))
                for i in range(n))
    relative_gap = max(0.0, primal - dual) / (1 + abs(primal))
    verified = error <= max(1e-3, 10 * precision) and relative_gap <= max(1e-4, 10 * precision)
    if trace is not None:
        record_trace(gap, offset)
    included = tuple(i for i in range(n) if abs(signed[i]) > 1e-8)
    return EpsilonTube(tuple(records[i] for i in included), tuple(signed[i] for i in included),
                       offset, geometry, penalty, tube_width, updates, gap, error,
                       primal, dual, relative_gap, verified)


def measure_epsilon_tube(tube, observations, *, progress=None):
    width = len(tube.retained_examples[0]) if tube.retained_examples else None
    result = []
    cases = observations if progress is None else progress(
        observations, desc="SVR predictions", unit="sample")
    for row in cases:
        candidate = tuple(float(v) for v in row)
        if width is not None and len(candidate) != width:
            raise ValueError('Dimension mismatch in regression observation.')
        result.append(tube.offset + sum(
            multiplier * _compare_rows(stored, candidate, tube.geometry)
            for stored, multiplier in zip(tube.retained_examples, tube.signed_weights)))
    return tuple(result)


## Usage

`construct_epsilon_tube(observations, targets, tube_width=0.1)` fits the regression model. `measure_epsilon_tube(tube, new_observations)` evaluates it at new inputs. Notebook 06 applies the solver to NHANES anthropometric measurements.
